In [1]:
import pandas as pd
df = pd.read_csv('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/predict_site_withec.csv')
substrates = df['SMILES'].to_list()
ecs = df['ecs'].to_list()
# dict_sub2ec = dict(zip(substrates,ecs))
ecs = [i.strip() for i in ecs]

# 凡是没有EC的我们用1.14代替
ecs = [i.replace('None','1.14') for i in ecs]
ecs = [i.split(' ') for i in ecs]
ecs_new = []
for eclist in ecs:
    eclist = [tuple([int(j) for j in i.split('.')[:2]]) for i in eclist]
    eclist = list(set(eclist))
    ecs_new.append(eclist)

subs_new = []
for i in range(len(substrates)):
    ecs = ecs_new[i]
    num_repeat = len(ecs)
    subs = [substrates[i] for _ in range(num_repeat)]
    subs_new = subs_new + subs

ecs_new = [i for k in ecs_new for i in k]

df_new = pd.DataFrame(columns=['substrate','ec','score'])
df_new['substrate'] = subs_new
df_new['ec'] = ecs_new
df_new['score'] = ['None' for _ in range(len(subs_new))]
df_new


,substrate,ec,score
0,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,"(1, 14)",None
1,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,"(2, 4)",None
2,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,"(1, 14)",None
3,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1,"(1, 14)",None
4,CC(CN1c2ccccc2Sc2ccccc21)N(C)C,"(1, 14)",None
...,...,...,...
67,O=C(O)c1cc(O)c2c(c1)C(C1c3cc(C(=O)O)cc(O)c3C(=...,"(1, 14)",None
68,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,"(1, 14)",None
69,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,"(2, 4)",None
70,O=P1(NCCCl)OCCCN1CCCl,"(1, 14)",None


In [2]:
from tqdm import tqdm

import json
from os.path import isfile

import torch
from torch_geometric.data import Data
from rdkit.Chem import Draw
import requests

from IPython.display import SVG

from gnn_som import createGnnSom, loadGnnSomState
from gnn_som.MolFromKcf import MolFromKcfFile

import os

from rdkit import Chem
from rdkit.Chem import AllChem

os.environ['CUDA_VISIBLE_DEVICES'] = "1"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open('data/config.json', 'r') as f:
    config = json.load(f)
config['features']['enzyme'] = [tuple(ec) for ec in config['features']['enzyme']] 

models = []
for i, params in enumerate(config['models']):
    model = createGnnSom(*config['models'][i])
    loadGnnSomState(model, torch.load('data/model%d.pt' % i, map_location=torch.device('cpu')))
    models.append(model)

# Convert SMILES to .mol file using RDKit 

atom_scores_list = []

for i in tqdm(range(len(df_new))):
    smi = df_new.iloc[i,0]
    enzyme = df_new.iloc[i,1]


    # Create RDKit mol object
    # molecule = 'CC1([C@@H]2[C@H]1[C@H](N(C2)C(=O)[C@H](C(C)(C)C)NC(=O)C(F)(F)F)C(=O)N[C@@H](C[C@@H]3CCNC3=O)C#N)C'
    print(smi)
    molecule = Chem.MolFromSmiles(smi)
    molecule = Chem.AddHs(molecule)

    if os.path.exists("molecule.mol"):
        os.remove("molecule.mol")

    # Compute 2D coordenates and write to .mol file
    AllChem.Compute2DCoords(molecule)
    print(Chem.MolToMolBlock(molecule), file=open("molecule.mol", 'a+'))

    if os.path.exists("molecule.kcf"):
        os.remove("molecule.kcf")

    # Convert .mol to .kcf (KEGG format used in the paper) 
    # https://iwatobipen.wordpress.com/2016/12/23/convert-chemical-file-format/
    # https://www.genome.jp/tools/gn_ca_tools_api.html
    !curl -F molfile=@molecule.mol http://rest.genome.jp/mol2kcf/ > molecule.kcf

    # Define mol
    mol = MolFromKcfFile('molecule.kcf')

        # enzyme = (2,4)

    print(enzyme)
    numFeatures = sum(len(feature) for feature in config['features'].values())
    x = torch.zeros((mol.GetNumAtoms(), numFeatures), dtype=torch.float32)
    for atom in mol.GetAtoms():
        x[atom.GetIdx(), config['features']['enzyme'].index(enzyme)] = 1
        offset = len(config['features']['enzyme'])
        x[atom.GetIdx(), offset + config['features']['element'].index(atom.GetSymbol())] = 1
        offset += len(config['features']['element'])
        x[atom.GetIdx(), offset + config['features']['kcfType'].index(atom.GetProp('kcfType'))] = 1

    edgeIndex = torch.zeros((2, mol.GetNumBonds() * 2), dtype=torch.int64)
    for bond in mol.GetBonds():
        i = bond.GetIdx()
        edgeIndex[0][i * 2] = bond.GetBeginAtomIdx()
        edgeIndex[1][i * 2] = bond.GetEndAtomIdx()
        edgeIndex[0][i * 2 + 1] = bond.GetEndAtomIdx()
        edgeIndex[1][i * 2 + 1] = bond.GetBeginAtomIdx()

    data = Data(x=x, edgeIndex=edgeIndex)

    # 确保模型和数据都在同一设备上运行
    data = data.to(device)  # 将数据移动到 GPU 或保持在 CPU
    y = None

    for model in models:
        model = model.to(device)  # 将每个模型移动到 GPU（如果尚未移动）
        newY = torch.sigmoid(model(data.x, data.edgeIndex))  # 数据已在 GPU 或 CPU 上
        y = newY if y is None else torch.add(y, newY)

    # 计算模型平均输出
    y = torch.div(y, len(models))
    print(y.shape)

    # import torch

    # # 假设 scores 是你的 torch 张量，大小是 [14, 1]
    # scores = y  # 示例：14个原子的反应分数，实际使用时替换为你的数据

    # # 创建一个空字典来存储原子序号和对应的分数
    # atom_scores = {}

    # # 将分数映射到原子序号
    # for idx, score in enumerate(scores):
    #     atom_scores[idx] = score.item()  # 使用 item() 将张量转换为标量

    # 打印结果
    # print(atom_scores)

    atom_scores_list.append(y)

    print(f'substrate: {smi}, ec: {enzyme}, score: {y}')

/tmp/ipykernel_352752/2810550191.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loadGnnSomState(model, torch.load('data/model%d.pt' % i, map_location=torch.device('cpu

NameError: name 'df_new' is not defined